# Table 1 — Stroke Cohort Summary

Publication-ready cohort characteristics table.

In [ ]:
import pandas as pd
import numpy as np

static = pd.read_parquet('../outputs/cohort/static_features.parquet')
print(f'Total patients: {len(static):,}')

In [ ]:
def table1_row_continuous(df, col, name=None):
    vals = df[col].dropna()
    return f'{name or col}: {vals.median():.1f} ({vals.quantile(0.25):.1f}-{vals.quantile(0.75):.1f})'

def table1_row_categorical(df, col, name=None):
    counts = df[col].value_counts()
    lines = [f'{name or col}:']
    for cat, n in counts.items():
        lines.append(f'  {cat}: {n} ({n/len(df)*100:.1f}%)')
    return '\n'.join(lines)

def table1_row_binary(df, col, name=None):
    n = df[col].sum()
    return f'{name or col}: {int(n)} ({n/len(df)*100:.1f}%)'

In [ ]:
# Generate Table 1
print('=' * 60)
print('TABLE 1: Cohort Characteristics')
print('=' * 60)
print(f'N = {len(static):,}')
print()
print('DEMOGRAPHICS')
print(table1_row_continuous(static, 'anchor_age', 'Age, median (IQR)'))
print(table1_row_categorical(static, 'gender', 'Sex'))
print(table1_row_categorical(static, 'race', 'Race/Ethnicity'))
print(table1_row_categorical(static, 'insurance', 'Insurance'))
print()
print('STROKE CHARACTERISTICS')
print(table1_row_categorical(static, 'stroke_subtype', 'Stroke Subtype'))
print(table1_row_continuous(static, 'los', 'ICU LOS days, median (IQR)'))
print(table1_row_binary(static, 'hospital_expire_flag', 'In-Hospital Mortality'))
print()
print('COMORBIDITIES')
for col in ['has_hypertension', 'has_diabetes', 'has_afib', 'has_dyslipidemia', 'has_ckd', 'has_cad']:
    print(table1_row_binary(static, col, col.replace('has_', '').replace('_', ' ').title()))
print()
print('ADMISSION LABS (first 24h)')
for col in [c for c in static.columns if c.endswith('_admit')]:
    print(table1_row_continuous(static, col, col.replace('_admit', '').title()))

In [ ]:
# Stratified by stroke subtype
print('\n' + '=' * 60)
print('TABLE 2: Stratified by Stroke Subtype')
print('=' * 60)
for subtype in static['stroke_subtype'].unique():
    subset = static[static['stroke_subtype'] == subtype]
    print(f'\n--- {subtype.upper()} (n={len(subset)}) ---')
    print(table1_row_continuous(subset, 'anchor_age', 'Age'))
    print(table1_row_binary(subset, 'hospital_expire_flag', 'Mortality'))
    print(table1_row_continuous(subset, 'los', 'ICU LOS'))